In [ ]:
# ===================================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ===================================================================
# NFStream es la librería clave para analizar los archivos .pcap
# El resto son para manipulación de datos, IA y gráficos.
# El -q es para que la salida de la instalación sea más limpia (quiet).
# ===================================================================
!pip install -q nfstream pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# ===================================================================
# CELDA 2: IMPORTACIÓN DE MÓDULOS
# ===================================================================
import os
import pandas as pd
import numpy as np
import re
from datetime import datetime
from nfstream import NFStreamer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ignoramos advertencias para mantener la salida limpia
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")


In [ ]:
# ===================================================================
# CELDA 3: FUNCIÓN INTELIGENTE DE DESFASE
# ===================================================================
def obtener_desfase_por_csv(archivo_sync, archivo_csv):
    """
    Calcula el desfase temporal entre el timestamp del juego y el timestamp UNIX
    buscando una marca de sincronización en un archivo de log.
    """
    print(f"   -> Buscando sincronización para {archivo_csv}...")
    
    # Extrae la fecha y hora del nombre del fichero de telemetría (ej: YYYYMMDD_HHMMSS)
    match_csv = re.search(r"(\d{8}_\d{6})", archivo_csv)
    if not match_csv:
        raise ValueError("El nombre del CSV no tiene el formato de fecha esperado (YYYYMMDD_HHMMSS).")
    fecha_csv = match_csv.group(1)
    
    with open(archivo_sync, 'r') as f:
        for linea in f: # Buscar linea por linea en el archivo de sincronización
            patron = r"\[([\d\.]+)\] ==== \[SYNC_TELEMETRIA\] Tiempo interno: ([\d\.]+) ===="
            match = re.search(patron, linea) # Extrae el tiempo UNIX y el tiempo del juego
            if match:
                tiempo_unix = float(match.group(1))
                tiempo_juego_sec = float(match.group(2))
                
                # Compara si la fecha y hora del log coincide con la del archivo CSV
                fecha_unix_str = datetime.fromtimestamp(tiempo_unix).strftime("%Y%m%d_%H%M%S")
                if fecha_unix_str == fecha_csv:
                    desfase = tiempo_unix - tiempo_juego_sec
                    print(f"      [OK] Sincronización encontrada. Desfase: {desfase:.3f} s")
                    return desfase
                    
    raise ValueError(f"No se encontró en '{archivo_sync}' la fecha correspondiente al CSV '{archivo_csv}'")
